In [1]:
import pandas as pd
import numpy as np
import os

# --- Configuration ---
# Path within your Docker Jupyter environment
ZEEK_LOG_PATH = '/home/jovyan/work/sample_conn.log'

# --- Check/Create Sample Log ---
if not os.path.exists(ZEEK_LOG_PATH):
    print("Creating a sample Zeek log file for the lab...")
    dummy_log_content = """#fields	ts	uid	id.orig_h	id.orig_p	id.resp_h	id.resp_p	proto	duration	orig_bytes	resp_bytes	conn_state	service	flag	orig_pkts	resp_pkts	tunnel	orig_ip_tos	resp_ip_tos
1678886400.123456	C1aB2c	192.168.1.100	54321	10.0.0.5	80	tcp	15.5	1024	5120	SF	http	SFT_SH	10	25	-	0x00	0x00
1678886405.678901	D2eF3g	192.168.1.100	54322	10.0.0.5	443	tcp	22.1	2048	8192	SF	https	SFT_SH	15	30	-	0x00	0x00
1678886410.112233	E3fG4h	192.168.1.101	54323	10.0.0.6	8080	tcp	5.0	512	2048	SF	http	SFT_SH	5	10	-	0x00	0x00
1678886415.445566	F4gH5i	192.168.1.100	54324	10.0.0.7	53	udp	0.1	64	128	SH	dns	SH	1	1	-	0x00	0x00
1678886420.778899	G5hI6j	192.168.1.102	54325	10.0.0.8	22	tcp	30.2	4096	16384	SF	ssh	SFT_SH	20	40	-	0x00	0x00
"""
    with open(ZEEK_LOG_PATH, 'w') as f:
        f.write(dummy_log_content)

# Load the log (Zeek uses tabs)
df_zeek = pd.read_csv(ZEEK_LOG_PATH, sep='\t', comment='#', names=['ts', 'uid', 'id.orig_h', 'id.orig_p', 'id.resp_h', 'id.resp_p', 'proto', 'duration', 'orig_bytes', 'resp_bytes', 'conn_state', 'service', 'flag', 'orig_pkts', 'resp_pkts', 'tunnel', 'orig_ip_tos', 'resp_ip_tos'])
print("Zeek log loaded successfully. Shape:", df_zeek.shape)
df_zeek.head()

Creating a sample Zeek log file for the lab...
Zeek log loaded successfully. Shape: (5, 18)


,ts,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,duration,orig_bytes,resp_bytes,conn_state,service,flag,orig_pkts,resp_pkts,tunnel,orig_ip_tos,resp_ip_tos
0,1.678886e+09,C1aB2c,192.168.1.100,54321,10.0.0.5,80,tcp,15.5,1024,5120,SF,http,SFT_SH,10,25,-,0x00,0x00
1,1.678886e+09,D2eF3g,192.168.1.100,54322,10.0.0.5,443,tcp,22.1,2048,8192,SF,https,SFT_SH,15,30,-,0x00,0x00
2,1.678886e+09,E3fG4h,192.168.1.101,54323,10.0.0.6,8080,tcp,5.0,512,2048,SF,http,SFT_SH,5,10,-,0x00,0x00
3,1.678886e+09,F4gH5i,192.168.1.100,54324,10.0.0.7,53,udp,0.1,64,128,SH,dns,SH,1,1,-,0x00,0x00
4,1.678886e+09,G5hI6j,192.168.1.102,54325,10.0.0.8,22,tcp,30.2,4096,16384,SF,ssh,SFT_SH,20,40,-,0x00,0x00


In [2]:
# Select relevant columns
features_to_extract = ['duration', 'proto', 'id.resp_p', 'service', 'flag']
df_features = df_zeek[features_to_extract].copy()

# 1. Clean Duration: Convert '-' to 0
df_features['duration'] = pd.to_numeric(df_features['duration'], errors='coerce').fillna(0)

# 2. Categorical Prep: Ensure strings and handle missing values
for col in ['proto', 'service', 'flag']:
    df_features[col] = df_features[col].astype(str).replace(['-', '(empty)'], 'unknown')

# 3. Port: Ensure integer
df_features['id.resp_p'] = pd.to_numeric(df_features['id.resp_p'], errors='coerce').fillna(0).astype(int)

print("Features cleaned and ready for encoding.")
df_features.head()

Features cleaned and ready for encoding.


,duration,proto,id.resp_p,service,flag
0,15.5,tcp,80,http,SFT_SH
1,22.1,tcp,443,https,SFT_SH
2,5.0,tcp,8080,http,SFT_SH
3,0.1,udp,53,dns,SH
4,30.2,tcp,22,ssh,SFT_SH


In [3]:
# Apply One-Hot Encoding
categorical_cols = ['proto', 'service', 'flag']
df_encoded = pd.get_dummies(df_features, columns=categorical_cols, prefix=categorical_cols)

print("Shape after encoding:", df_encoded.shape)
# Displaying specific encoded columns for verification
df_encoded.filter(like='proto_').head()

Shape after encoding: (5, 10)


,proto_tcp,proto_udp
0,True,False
1,True,False
2,True,False
3,False,True
4,True,False


In [4]:
# --- Advanced Feature Engineering ---

# 1. Connection Rate: Calculate the number of connections per second
# Note: This requires the 'ts' column from the original loading step
df_encoded['ts'] = df_zeek['ts']
df_encoded = df_encoded.sort_values('ts')
df_encoded['time_diff'] = df_encoded['ts'].diff().fillna(0)

# Avoid division by zero for identical timestamps
df_encoded['connection_rate'] = 1 / df_encoded['time_diff'].replace(0, np.nan)
df_encoded['connection_rate'] = df_encoded['connection_rate'].fillna(0)

# 2. Byte Transfer Rate: Calculate bytes transferred per second
# We use the original bytes columns we loaded in Step 2
df_encoded['orig_bytes'] = df_zeek['orig_bytes'].apply(pd.to_numeric, errors='coerce').fillna(0)
df_encoded['resp_bytes'] = df_zeek['resp_bytes'].apply(pd.to_numeric, errors='coerce').fillna(0)

df_encoded['orig_bytes_per_sec'] = df_encoded['orig_bytes'] / df_encoded['duration'].replace(0, np.nan)
df_encoded['resp_bytes_per_sec'] = df_encoded['resp_bytes'] / df_encoded['duration'].replace(0, np.nan)

# Clean up NaN results from duration=0
df_encoded['orig_bytes_per_sec'] = df_encoded['orig_bytes_per_sec'].fillna(0)
df_encoded['resp_bytes_per_sec'] = df_encoded['resp_bytes_per_sec'].fillna(0)

print("Advanced features (Rates) calculated successfully.")
df_encoded[['duration', 'connection_rate', 'orig_bytes_per_sec', 'resp_bytes_per_sec']].head()

Advanced features (Rates) calculated successfully.


,duration,connection_rate,orig_bytes_per_sec,resp_bytes_per_sec
0,15.5,0.000000,66.064516,330.322581
1,22.1,0.180004,92.669683,370.678733
2,5.0,0.225564,102.400000,409.600000
3,0.1,0.187500,640.000000,1280.000000
4,30.2,0.187500,135.629139,542.516556


In [5]:
# Exclude identifier columns that aren't useful for AI training
identifier_cols = ['ts', 'uid', 'id.orig_h', 'id.orig_p', 'id.resp_h', 'conn_state', 'tunnel', 'orig_ip_tos', 'resp_ip_tos', 'time_diff', 'orig_bytes', 'resp_bytes']
final_feature_columns = [col for col in df_encoded.columns if col not in identifier_cols]

df_final_features = df_encoded[final_feature_columns]

print(f"Final feature set shape: {df_final_features.shape}")
print("\nFirst 5 rows of AI-Ready data:")
df_final_features.head()

Final feature set shape: (5, 13)

First 5 rows of AI-Ready data:


,duration,id.resp_p,proto_tcp,proto_udp,service_dns,service_http,service_https,service_ssh,flag_SFT_SH,flag_SH,connection_rate,orig_bytes_per_sec,resp_bytes_per_sec
0,15.5,80,True,False,False,True,False,False,True,False,0.000000,66.064516,330.322581
1,22.1,443,True,False,False,False,True,False,True,False,0.180004,92.669683,370.678733
2,5.0,8080,True,False,False,True,False,False,True,False,0.225564,102.400000,409.600000
3,0.1,53,False,True,True,False,False,False,False,True,0.187500,640.000000,1280.000000
4,30.2,22,True,False,False,False,False,True,True,False,0.187500,135.629139,542.516556
